# Fig. 4 

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

SCI_STYLE = {
    'font.family': 'Arial',
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'figure.dpi': 900,
    'savefig.format': 'tiff',
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
}

plt.rcParams.update(SCI_STYLE)

df = pd.read_csv("../final_report1.csv")

custom_study_order = [
    'ALMANAC', 'FRIEDMAN', 'ONEIL', 'ASTRAZENECA', 'NCATS_ES(NAMPT+PARP)',
    'NCATS_HL', 'NCATS_ES(FAKI_AURKI)', 'FORCINA', 'FLOBAK', 'YOHE', 'DYALL',
    'FALLAHI-SICHANI', 'NCATS_2D_3D'
]

custom_model_order = [
    'JointSyn', 'CGMS', 'DeepDDS', 'MTLSynergy', 'MLP', 'XGB', 'RF'
]

target_mapping = {
    'synergy_loewe': 'Loewe',
    'synergy_hsa': 'HSA',
    'synergy_bliss': 'Bliss',
    'synergy_zip': 'ZIP'
}

metrics_config = {
    'PCC': {
        'column': 'PCC_Mean',
        'vmin': 0,
        'vmax': 1,
        'cmap': 'YlGnBu'
    },
    'R2': {
        'column': 'R2_Mean',
        'vmin': 0,
        'vmax': 1,
        'cmap': 'YlGnBu'
    },
    'MSE': {
        'column': 'MSE_Mean',
        'vmin': 0,
        'vmax': 300,
        'cmap': 'YlGnBu_r'
    },
    'RMSE': {
        'column': 'RMSE_Mean',
        'vmin': 0,
        'vmax': df['RMSE_Mean'].quantile(0.95),
        'cmap': 'YlGnBu_r'
    },
    'MAE': {
        'column': 'MAE_Mean',
        'vmin': 0,
        'vmax': df['MAE_Mean'].quantile(0.95),
        'cmap': 'YlGnBu_r'
    }
}

for metric_name, config in metrics_config.items():
    n_rows = len(custom_study_order)
    n_cols = len(custom_model_order)

    fig, axs = plt.subplots(2, 2, figsize=(7.5, 4.5), 
                           sharex=True, sharey=True,
                           gridspec_kw={'wspace': 0.05, 'hspace': 0.15})

    sorted_targets = sorted(df['Target'].unique())

    for i, target in enumerate(sorted_targets):
        row_idx = i // 2
        col_idx = i % 2

        pivot_df = df[df['Target'] == target].pivot_table(
            index='Study',
            columns='Model',
            values=config['column'],
            aggfunc='mean'
        ).reindex(index=custom_study_order, columns=custom_model_order)

        pivot_df.index.name = None
        pivot_df.columns.name = None

        annot_array = np.empty_like(pivot_df.values, dtype=object)

        text_colors = np.empty_like(pivot_df.values, dtype=object)

        norm = plt.Normalize(vmin=config['vmin'], vmax=config['vmax'])
        cmap = plt.get_cmap(config['cmap'])
        
        for i_row in range(pivot_df.shape[0]):
            for i_col in range(pivot_df.shape[1]):
                value = pivot_df.iloc[i_row, i_col]
                if pd.isna(value):
                    annot_array[i_row, i_col] = ""
                    text_colors[i_row, i_col] = "black"  
                else:
                    if value > config['vmax']:
                        annot_array[i_row, i_col] = f"{config['vmax']:.0f}+" if config['vmax'] > 10 else f">{config['vmax']:.2f}"
                    else:
                        if abs(value) > 10:
                            annot_array[i_row, i_col] = f"{value:.0f}"
                        else:
                            annot_array[i_row, i_col] = f"{value:.2f}"

                    norm_value = norm(value) if value <= config['vmax'] else 1.0
                    rgba = cmap(norm_value)
                    brightness = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
                    text_colors[i_row, i_col] = "white" if brightness < 0.6 else "black"

        sns.heatmap(
            pivot_df,
            annot=annot_array, 
            fmt="",
            cmap=config['cmap'],
            vmin=config['vmin'],
            vmax=config['vmax'],
            cbar=False,  
            linewidths=0.5,
            annot_kws={"size": 7},  
            square=False,
            ax=ax
        )

        for i_row in range(pivot_df.shape[0]):
            for i_col in range(pivot_df.shape[1]):
                if not pd.isna(pivot_df.iloc[i_row, i_col]):
                    text = ax.texts[i_row * pivot_df.shape[1] + i_col]
                    text.set_color(text_colors[i_row, i_col])

        mapped_target = target_mapping.get(target, target)
        ax.set_title(f"{mapped_target}", fontsize=8, pad=6)
        ax.set_xticklabels(custom_model_order, rotation=45, ha='right', fontsize=7)
        ax.set_yticklabels(custom_study_order, rotation=0, fontsize=7)

    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7]) 
    sm = plt.cm.ScalarMappable(cmap=config['cmap'], 
                              norm=plt.Normalize(vmin=config['vmin'], vmax=config['vmax']))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)

    if metric_name == 'R2':
        cbar_label = r'$\mathrm{R}^2$'
    else:
        cbar_label = metric_name
        
    cbar.set_label(cbar_label, fontsize=8, labelpad=5)
    cbar.ax.tick_params(labelsize=7)   

    plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)

    plt.savefig(f'{metric_name}_performance_heatmap_by_datasets.tiff', dpi=900, bbox_inches='tight', facecolor='white')
    plt.close(fig)



C:\Users\84991\AppData\Local\Temp\ipykernel_56436\3821169043.py:197: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ipykernel_56436\3821169043.py:197: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ipykernel_56436\3821169043.py:197: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ipykernel_56436\3821169043.py:197: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ip

所有数据集热力图已按SCI要求生成


# Fig. 5

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

SCI_STYLE = {
    'font.family': 'Arial',
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'figure.dpi': 900,
    'savefig.format': 'tiff',
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
}

plt.rcParams.update(SCI_STYLE)

df = pd.read_csv("../final_report1.csv")

custom_study_order = [
    'ALMANAC', 'FRIEDMAN', 'ONEIL', 'ASTRAZENECA', 'NCATS_ES(NAMPT+PARP)',
    'NCATS_HL', 'NCATS_ES(FAKI_AURKI)', 'FORCINA', 'FLOBAK', 'YOHE', 'DYALL',
    'FALLAHI-SICHANI', 'NCATS_2D_3D'
]

custom_model_order = [
    'JointSyn', 'CGMS', 'DeepDDS', 'MTLSynergy', 'MLP', 'XGB', 'RF'
]

target_mapping = {
    'synergy_loewe': 'Loewe',
    'synergy_hsa': 'HSA',
    'synergy_bliss': 'Bliss',
    'synergy_zip': 'ZIP'
}

metrics_config = {
    'PCC': {
        'column': 'PCC_Mean',
        'vmin': 0,
        'vmax': 1,
        'cmap': 'YlGnBu'
    },
    'R2': {
        'column': 'R2_Mean',
        'vmin': 0,
        'vmax': 1,
        'cmap': 'YlGnBu'
    },
    'MSE': {
        'column': 'MSE_Mean',
        'vmin': 0,
        'vmax': 300,
        'cmap': 'YlGnBu_r'
    },
    'RMSE': {
        'column': 'RMSE_Mean',
        'vmin': 0,
        'vmax': df['RMSE_Mean'].quantile(0.95),
        'cmap': 'YlGnBu_r'
    },
    'MAE': {
        'column': 'MAE_Mean',
        'vmin': 0,
        'vmax': df['MAE_Mean'].quantile(0.95),
        'cmap': 'YlGnBu_r'
    }
}

for metric_name, config in metrics_config.items():
    n_rows = len(custom_model_order)
    n_cols = len(custom_study_order)

    fig, axs = plt.subplots(2, 2, figsize=(7.5, 4.5), 
                           sharex=True, sharey=True,
                           gridspec_kw={'wspace': 0.05, 'hspace': 0.15})

    sorted_targets = sorted(df['Target'].unique())

    for i, target in enumerate(sorted_targets):
        row_idx = i // 2
        col_idx = i % 2
        ax = axs[row_idx, col_idx]

        pivot_df = df[df['Target'] == target].pivot_table(
            index='Model',
            columns='Study',
            values=config['column'],
            aggfunc='mean'
        ).reindex(index=custom_model_order, columns=custom_study_order)

        pivot_df.index.name = None
        pivot_df.columns.name = None

        annot_array = np.empty_like(pivot_df.values, dtype=object)
        text_colors = np.empty_like(pivot_df.values, dtype=object)

        norm = plt.Normalize(vmin=config['vmin'], vmax=config['vmax'])
        cmap = plt.get_cmap(config['cmap'])
        
        for i_row in range(pivot_df.shape[0]):
            for i_col in range(pivot_df.shape[1]):
                value = pivot_df.iloc[i_row, i_col]
                if pd.isna(value):
                    annot_array[i_row, i_col] = ""
                    text_colors[i_row, i_col] = "black" 
                else:
                    if value > config['vmax']:
                        annot_array[i_row, i_col] = f"{config['vmax']:.0f}+" if config['vmax'] > 10 else f">{config['vmax']:.2f}"
                    else:
                        if abs(value) > 10:
                            annot_array[i_row, i_col] = f"{value:.0f}"
                        else:
                            annot_array[i_row, i_col] = f"{value:.2f}"

                    norm_value = norm(value) if value <= config['vmax'] else 1.0
                    rgba = cmap(norm_value)
                    brightness = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
                    text_colors[i_row, i_col] = "white" if brightness < 0.6 else "black"

        sns.heatmap(
            pivot_df,
            annot=annot_array, 
            fmt="",
            cmap=config['cmap'],
            vmin=config['vmin'],
            vmax=config['vmax'],
            cbar=False,  
            linewidths=0.5,
            annot_kws={"size": 7}, 
            square=False,
            ax=ax
        )

        for i_row in range(pivot_df.shape[0]):
            for i_col in range(pivot_df.shape[1]):
                if not pd.isna(pivot_df.iloc[i_row, i_col]):
                    text = ax.texts[i_row * pivot_df.shape[1] + i_col]
                    text.set_color(text_colors[i_row, i_col])

        mapped_target = target_mapping.get(target, target)
        ax.set_title(f"{mapped_target}", fontsize=8, pad=6)

        ax.set_xticklabels(custom_study_order, rotation=45, ha='right', fontsize=7)
        ax.set_yticklabels(custom_model_order, rotation=0, fontsize=7)

    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7]) 
    sm = plt.cm.ScalarMappable(cmap=config['cmap'], 
                              norm=plt.Normalize(vmin=config['vmin'], vmax=config['vmax']))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
 
    if metric_name == 'R2':
        cbar_label = r'$\mathrm{R}^2$'
    else:
        cbar_label = metric_name
        
    cbar.set_label(cbar_label, fontsize=8, labelpad=5)
    cbar.ax.tick_params(labelsize=7)

    plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)

    plt.savefig(f'{metric_name}_model_rankings_heatmap.tiff', dpi=900, bbox_inches='tight', facecolor='white')
    plt.close(fig)



C:\Users\84991\AppData\Local\Temp\ipykernel_56436\3057494092.py:197: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ipykernel_56436\3057494092.py:197: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ipykernel_56436\3057494092.py:197: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ipykernel_56436\3057494092.py:197: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ip

所有模型排名热力图已按SCI要求生成
